# Gradient Descent Structure Refinement
This example demonstrates how to use `diff-biophys` and `optax` to refine a structure
directly against experimental data using gradient descent.

We will simulate a target "experimental" Cryo-EM map and try to optimize a random
starting map to maximize the Fourier Shell Correlation (FSC) with the target.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/elkins/diff-biophys/blob/main/examples/interactive_tutorials/structure_refinement.ipynb)
\n

In [ ]:
import sys
if "google.colab" in sys.modules:
    !pip install -q diff-biophys py3Dmol biotite matplotlib optax
else:
    sys.path.append("../../")

import matplotlib.pyplot as plt
import numpy as np
import biotite.structure.io as strucio


In [ ]:
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import optax

from diff_biophys.cryo_em import compute_fsc

## 1. Setup the Target Data
Let's create a dummy target density map. In a real scenario, this would be your
experimental Cryo-EM map loaded via `mrcfile`.

In [ ]:
shape = (16, 16, 16)
voxel_size = (1.0, 1.0, 1.0)
key = jax.random.PRNGKey(42)

# Generate a smooth target map
target_map = jax.random.normal(key, shape)
target_map = jax.scipy.ndimage.map_coordinates(target_map, tuple(jnp.indices(shape) * 0.5), order=1)

## 2. Initialize the Model
We'll initialize our "predicted" map with random noise.
Our goal is to update this map to match the target.

In [ ]:
key, subkey = jax.random.split(key)
params = jax.random.normal(subkey, shape)

## 3. Define the Loss Function
We want to maximize the FSC. Since `optax` minimizes the loss, we return the negative
sum of the FSC curve across all valid frequency bins.

In [ ]:
@jax.jit
def loss_fn(pred_map: jax.Array, target: jax.Array) -> jax.Array:
    freqs, fsc = compute_fsc(pred_map, target, voxel_size)

    # We only want to sum over valid frequencies (where freqs > 0)
    # The jax FSC function pads invalid bins with 0s for frequencies.
    valid_mask = freqs > 0.0

    # Negative FSC sum for minimization
    return -jnp.sum(jnp.where(valid_mask, fsc, 0.0))

## 4. Setup the Optimizer
We use `optax.adam` with a learning rate of 0.1.

In [ ]:
optimizer = optax.adam(learning_rate=0.1)
opt_state = optimizer.init(params)

## 5. Optimization Loop
We compute the value and gradient of the loss function, and update the parameters.

In [ ]:
@jax.jit
def step(
    params: jax.Array, opt_state: optax.OptState, target: jax.Array
) -> tuple[jax.Array, optax.OptState, jax.Array]:
    loss_value, grads = jax.value_and_grad(loss_fn)(params, target)
    updates, opt_state = optimizer.update(grads, opt_state, params)
    params = optax.apply_updates(params, updates)
    return params, opt_state, loss_value


print("Starting Refinement...")
losses = []
for i in range(50):
    params, opt_state, loss_value = step(params, opt_state, target_map)
    losses.append(loss_value)
    if i % 10 == 0:
        print(f"Step {i:03d} | Loss: {loss_value:.4f}")

## 6. Results
As you can see, the negative FSC loss decreases rapidly as the predicted map
converges to the target map!

In [ ]:
plt.plot(losses)
plt.title("FSC Optimization Loss")
plt.xlabel("Step")
plt.ylabel("Negative FSC Sum")
plt.savefig("fsc_optimization.png")
print("Optimization plot saved to fsc_optimization.png")
plt.show()

## 7. Fourier Shell Correlation (FSC) Curve

The 'gold standard' for determining the resolution of a Cryo-EM map is the FSC curve. The resolution is typically defined as the spatial frequency where the FSC drops below the 0.143 cutoff threshold.

Let's compute the final FSC between our optimized predicted map and the target map.

In [ ]:
import numpy as np
# Compute the final FSC curve
final_freqs, final_fsc = compute_fsc(params, target_map, voxel_size)

# Filter out zero frequencies
valid = np.array(final_freqs) > 0.0
freqs = np.array(final_freqs)[valid]
fsc = np.array(final_fsc)[valid]

plt.figure(figsize=(8, 5))
plt.plot(freqs, fsc, color='purple', linewidth=2.5, label='Final Map vs Target Map')
plt.axhline(0.143, color='red', linestyle='--', label='0.143 Cutoff')

plt.xlim(0, max(freqs))
plt.ylim(-0.1, 1.1)
plt.xlabel('Spatial Frequency (1/Å)', fontsize=12)
plt.ylabel('Fourier Shell Correlation (FSC)', fontsize=12)
plt.title('FSC Resolution Curve', fontsize=14)
plt.legend()
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()


## 8. Density Map Slices

Finally, let's visualize a 2D slice through the 3D density volume to see how well our predicted density matches the target density at the end of the refinement.

In [ ]:
# Extract a middle slice along the Z-axis
z_slice = 8

target_slice = np.array(target_map[z_slice, :, :])
pred_slice = np.array(params[z_slice, :, :])

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Target Map
im0 = axes[0].imshow(target_slice, cmap='magma', origin='lower')
axes[0].set_title('Target Map (Z=8)')
fig.colorbar(im0, ax=axes[0])

# Predicted Map
im1 = axes[1].imshow(pred_slice, cmap='magma', origin='lower')
axes[1].set_title('Optimized Predicted Map (Z=8)')
fig.colorbar(im1, ax=axes[1])

# Difference Map
diff_slice = target_slice - pred_slice
im2 = axes[2].imshow(diff_slice, cmap='bwr', origin='lower', vmin=-np.max(np.abs(diff_slice)), vmax=np.max(np.abs(diff_slice)))
axes[2].set_title('Difference Map')
fig.colorbar(im2, ax=axes[2])

plt.tight_layout()
plt.show()
